# Destek 03 — Göç kalkanı (Görsel 6)

**Amaç:** Hangi ülkelerin nüfusunu göçle ayakta tuttuğunu ve bu telafinin zamanla nasıl zayıfladığını gösteren dağılım grafiğinin verisini üretmek. Kontrol listesi #7'yi hesaplamak.

**Girdi:** `data/clean/countries_clean.csv`, 161 büyük ülke (2024 nüfusu > 1 milyon; `ulkeler_1m.csv` varsa o kullanılır), 1990–2100, 5 yılda bir. 2101 dışarıda.

**Eksenler:** x = doğal artış hızı `NatChangeRT` (binde), y = net göç hızı `CNMR` (binde), balon = nüfus, renk = bölge. Hızlar binde (1000 kişi başına) birimindedir.

**Çıktı:** `data/out/g06_goc_kalkani.csv` (grafik verisi), `data/out/g06_goc_kalkani_sayim.csv` (her yıl için dört bölgedeki ülke sayıları; slayttaki sayıların kaynağı).

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
OUT = ROOT / "data/out"; OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(ROOT / "data/clean/countries_clean.csv")
df = raw[raw.Time <= 2100].copy()                       # 2101 dışarıda

# --- 161 büyük ülke: tek tanım = data/clean/ulkeler_1m.csv (liderin dosyası); yoksa 2024 nüfusu > 1 milyon kuralı
hesap = set(df[(df.Time == 2024) & (df.TPopulation1July > 1000)].LocID)     # nüfus bin kişi
ulk = ROOT / "data/clean/ulkeler_1m.csv"
if ulk.exists():
    u = pd.read_csv(ulk)
    kol = {k.lower(): k for k in u.columns}                                    # sütun adı büyük/küçük harf farkını tolere et
    anahtar = next(k for k in ("locid", "iso3_code", "iso3") if k in kol)
    harita = {"locid": "LocID", "iso3_code": "ISO3_code", "iso3": "ISO3_code"}[anahtar]
    ids = set(df[df[harita].isin(u[kol[anahtar]])].LocID)
    print(f"ulkeler_1m.csv okundu ({len(u)} satır, eşleşme sütunu: {kol[anahtar]}).")
    print("Dosyadaki liste, 'nüfus > 1 milyon' kuralıyla aynı mı:", ids == hesap)
    if ids != hesap: print("Fark:", sorted(df[df.LocID.isin(ids ^ hesap)].Location.unique()))
else:
    ids = hesap; print("ulkeler_1m.csv bulunamadı; 2024 nüfusu > 1 milyon kuralıyla hesaplandı.")
assert len(ids) == 161, len(ids)
b = df[df.LocID.isin(ids)].copy()
print("Ülke:", b.LocID.nunique(), "| yıl:", b.Time.min(), "-", b.Time.max(), "| satır:", len(b))

ulkeler_1m.csv okundu (161 satır, eşleşme sütunu: iso3).
Dosyadaki liste, 'nüfus > 1 milyon' kuralıyla aynı mı: True
Ülke: 161 | yıl: 1950 - 2100 | satır: 24311


## 1. Dört bölge (quadrant) tanımı

| `quadrant` | Anlamı | Doğal artış (x) | Net göç (y) |
|---|---|---|---|
| `growth_inflow` | Sağ üst: doğal artış + göç alıyor | ≥ 0 | > 0 |
| `shield` | Sol üst: doğal azalış + göç alıyor ("göç kalkanı") | < 0 | > 0 |
| `double_loss` | Sol alt: doğal azalış + göç veriyor ("çifte kayıp") | < 0 | ≤ 0 |
| `growth_outflow` | Sağ alt: doğal artış + göç veriyor | ≥ 0 | ≤ 0 |

`pop_change_positive`: nüfusu o yıl yine de artıyor mu (`PopChange > 0`). "Göçle büyüyen" = doğal azalış var ama nüfus artıyor. Net göçü tam sıfır olan satırlar (yalnızca projeksiyonlarda) göç vermiyor sayılır (`≤ 0`).

In [2]:
def quadrant(nat, mig):
    return np.select(
        [(nat >= 0) & (mig > 0), (nat < 0) & (mig > 0), (nat < 0) & (mig <= 0)],
        ["growth_inflow", "shield", "double_loss"], default="growth_outflow")

b["quadrant"] = quadrant(b.NatChangeRT, b.CNMR)
b["pop_change_positive"] = b.PopChange > 0
b["nat_decline"] = b.NatChange < 0
print("Doğal azalıştaki satırlar içinde net göçü tam 0 olan:", int(((b.NatChangeRT < 0) & (b.CNMR == 0)).sum()))
print("Doğal azalış (hız) ile doğal azalış (sayı) işareti her satırda aynı mı:", bool((b.nat_decline == (b.NatChangeRT < 0)).all()))

Doğal azalıştaki satırlar içinde net göçü tam 0 olan: 3
Doğal azalış (hız) ile doğal azalış (sayı) işareti her satırda aynı mı: True


## 2. Kontrol sayıları (kontrol listesi #7)

In [3]:
def sayim(g):
    return pd.Series({
        "n_countries": len(g),
        "n_natural_decline": int((g.NatChange < 0).sum()),
        "n_shield": int((g.quadrant == "shield").sum()),
        "n_shield_growing": int(((g.quadrant == "shield") & g.pop_change_positive).sum()),
        "n_double_loss": int((g.quadrant == "double_loss").sum()),
        "n_growth_inflow": int((g.quadrant == "growth_inflow").sum()),
        "n_growth_outflow": int((g.quadrant == "growth_outflow").sum()),
        "n_decline_and_growing": int(((g.NatChange < 0) & g.pop_change_positive).sum()),
    })
yillik = b[b.Time >= 1990].groupby("Time").apply(sayim).astype(int)
yillik["is_projection"] = yillik.index >= 2024

beklenen = {2023: (35, 11), 2050: (61, 9), 2100: (101, 4)}
kt = pd.DataFrame([(y, int(yillik.loc[y, "n_natural_decline"]), int(yillik.loc[y, "n_decline_and_growing"]), *beklenen[y]) for y in beklenen],
                  columns=["yıl", "doğal_azalış", "göçle_büyüyen", "beklenen_azalış", "beklenen_büyüyen"])
kt["tamam"] = (kt.doğal_azalış == kt.beklenen_azalış) & (kt.göçle_büyüyen == kt.beklenen_büyüyen)
kt["büyüyen_payı_%"] = (kt.göçle_büyüyen / kt.doğal_azalış * 100).round(0)
display(kt)
# kalkan tanımının iki yolla aynı sonucu verdiğini göster: (doğal azalış & PopChange>0) = (kalkan bölgesi & PopChange>0)
print("Kalkan bölgesinde nüfusu artan = doğal azalışta nüfusu artan (tüm yıllar):", bool((yillik.n_shield_growing == yillik.n_decline_and_growing).all()))
display(yillik.loc[[1990, 2000, 2010, 2023, 2030, 2050, 2075, 2100]])

,yıl,doğal_azalış,göçle_büyüyen,beklenen_azalış,beklenen_büyüyen,tamam,büyüyen_payı_%
0,2023,35,11,35,11,True,31.0
1,2050,61,9,61,9,True,15.0
2,2100,101,4,101,4,True,4.0


Kalkan bölgesinde nüfusu artan = doğal azalışta nüfusu artan (tüm yıllar): True


,n_countries,n_natural_decline,n_shield,n_shield_growing,n_double_loss,n_growth_inflow,n_growth_outflow,n_decline_and_growing,is_projection
Time,,,,,,,,,
1990,161,4,2,1,2,67,90,1,False
2000,161,17,8,4,9,60,84,4,False
2010,161,15,8,2,7,62,84,2,False
2023,161,35,22,11,13,48,78,11,False
2030,161,45,22,7,23,40,76,7,True
2050,161,61,33,9,28,24,76,9,True
2075,161,81,34,5,47,25,55,5,True
2100,161,101,38,4,63,18,42,4,True


**Okuma:** Doğal azalış yaşayan ülke sayısı hızla artıyor, göçle büyümeyi sürdürebilenlerin payı ise küçülüyor. Bu "telafi zayıflıyor" mesajının sayısal karşılığı. Kalkan bölgesinde nüfusu artanlar ile doğal azalışta nüfusu artanlar her yıl aynı kümedir, yani tanım iki yoldan aynı sonucu veriyor.

## 3. Tutarlılık: `NatChange + NetMigrations ≈ PopChange`

In [4]:
b["tutarlilik_fark_bin"] = (b.NatChange + b.NetMigrations - b.PopChange).abs()
sorun = b[(b.Time >= 1990) & (b.tutarlilik_fark_bin > 1)][["Location", "Time", "NatChange", "NetMigrations", "PopChange", "tutarlilik_fark_bin"]]
print(f"161 ülke × 1990–2100: fark > 1 bin kişi olan ülke-yıl sayısı = {len(sorun)}; en büyük fark = {b[b.Time >= 1990].tutarlilik_fark_bin.max():.2f} bin kişi")
display(sorun.round(3))

161 ülke × 1990–2100: fark > 1 bin kişi olan ülke-yıl sayısı = 9; en büyük fark = 2.79 bin kişi


,Location,Time,NatChange,NetMigrations,PopChange,tutarlilik_fark_bin
16361,United Arab Emirates,2047,157.084,12.363,172.235,2.788
16362,United Arab Emirates,2048,157.978,13.714,173.884,2.192
16363,United Arab Emirates,2049,159.148,14.276,175.591,2.167
16364,United Arab Emirates,2050,160.225,13.841,175.964,1.898
16365,United Arab Emirates,2051,161.780,13.216,176.595,1.599
16366,United Arab Emirates,2052,163.547,13.390,178.014,1.077
16367,United Arab Emirates,2053,165.696,13.276,180.073,1.101
16368,United Arab Emirates,2054,168.082,12.993,182.766,1.691
16372,United Arab Emirates,2058,178.682,11.216,191.410,1.512


**Okuma:** Fark listesi yalnızca BAE'nin 2047–2058 projeksiyon yıllarını içeriyor. En büyük fark yaklaşık 3 bin kişi; BAE'nin yıllık nüfus değişimi o yıllarda 170 binin üzerinde, yani fark %2'nin altında. Veri tarafından açıklanamayan bir tutarsızlık yok, bu ülke-yıllar `note` sütununda ayrıca işaretlenmedi. Yuvarlama gürültüsü olarak kabul edildi.

## 4. Uç değerler ve `note` sütunu

Yol haritası kuralı: y ekseni **±20 binde** ile sınırlanır; bu eksenin dışında kalan noktalar `note` sütununda açıklanır, grafikte etiketlemeyi görselleştirmeci yapar. CSV'deki değerler **kırpılmaz**, gerçek değerlerdir.

Açıklamalar iki kaynaktan gelir: (1) bilinen olaylar (savaş, soykırım, bağımsızlık süreçleri, Sovyetler Birliği sonrası göç) ve Körfez ülkelerinin emek göçü, (2) geri kalanı için genel "uç değer" ifadesi. **Olay etiketleri genel tarihsel bilgiye dayanır; veri bunları kendisi söylemiyor.** Sunuma girmeden önce ikinci göz kontrol etmeli.

In [5]:
LIM = 20
OLAYLAR = [   # (ülkeler, ilk yıl, son yıl, açıklama)
 (["Syrian Arab Republic"],                           2012, 2016, "Suriye iç savaşı: kitlesel göç çıkışı"),
 (["Ukraine"],                                        2022, 2022, "Rusya'nın Ukrayna'yı işgali: kitlesel göç çıkışı"),
 (["Kuwait"],                                         1990, 1991, "Kuveyt'in işgali ve Körfez Savaşı"),
 (["Rwanda"],                                         1994, 1996, "Ruanda soykırımı ve sonrasındaki geri dönüşler"),
 (["Bosnia and Herzegovina"],                         1992, 1996, "Bosna Savaşı"),
 (["Kosovo (under UNSC res. 1244)"],                  1998, 1999, "Kosova Savaşı"),
 (["Timor-Leste"],                                    1999, 2002, "Doğu Timor bağımsızlık sürecindeki şiddet ve geri dönüşler"),
 (["Liberia"],                                        1990, 1997, "Liberya iç savaşı"),
 (["Somalia"],                                        1990, 1993, "Somali iç savaşı ve kıtlık"),
 (["Burundi"],                                        1993, 1996, "Burundi iç savaşı"),
 (["Burundi"],                                        2015, 2015, "Burundi siyasi krizi: mülteci çıkışı"),
 (["Afghanistan"],                                    1990, 2007, "Savaş dönemi mülteci çıkışları ve geri dönüşleri"),
 (["Armenia", "Georgia", "Kazakhstan", "Kyrgyzstan"], 1991, 2001, "Sovyetler Birliği sonrası göç çıkışı"),
]
GORFEZ = ["Qatar", "United Arab Emirates", "Kuwait", "Oman", "Bahrain", "Saudi Arabia"]

def not_yaz(r):
    if abs(r.CNMR) <= LIM: return ""
    for ulkeler, y0, y1, text in OLAYLAR:
        if r.Location in ulkeler and y0 <= r.Time <= y1: return f"eksen dışı (|net göç| > {LIM} binde): {text}"
    if r.Location in GORFEZ: return f"eksen dışı (|net göç| > {LIM} binde): Körfez ülkesi, emek göçü kaynaklı oynaklık"
    return f"eksen dışı (|net göç| > {LIM} binde): uç değer, neden ayrıca incelenmedi"

b["note"] = b.apply(not_yaz, axis=1)
GRID = list(range(1990, 2101, 5))
g = b[b.Time.isin(GRID)]
yd = b[(b.Time >= 1990) & (b.CNMR.abs() > LIM)]
print(f"|net göç| > {LIM} binde: yıllık veride {len(yd)} satır, 5 yıllık karelerde {int((g.CNMR.abs() > LIM).sum())} satır")
print("5 yıllık karelerde sınır dışı:", g[g.CNMR.abs() > LIM].LocID.nunique(), "ülke")
olay_disi = yd[(~yd.Time.isin(GRID)) & yd.note.str.contains("Suriye|Ukrayna|Kuveyt|Ruanda|Bosna|Kosova|Doğu Timor|Liberya|Somali|Burundi", regex=True)]
print("Olay etiketi taşıyıp 5 yıllık kareye denk gelmeyen ülke-yıllar (grafikte görünmez):")
display(olay_disi.groupby("Location").Time.apply(lambda s: ", ".join(map(str, s))).to_frame("yıllar"))
display(g[g.CNMR.abs() > LIM][["Location", "Time", "NatChangeRT", "CNMR", "note"]].sort_values(["Location", "Time"]).round(2).head(25))

|net göç| > 20 binde: yıllık veride 309 satır, 5 yıllık karelerde 62 satır
5 yıllık karelerde sınır dışı: 35 ülke
Olay etiketi taşıyıp 5 yıllık kareye denk gelmeyen ülke-yıllar (grafikte görünmez):


,yıllar
Location,
Bosnia and Herzegovina,"1992, 1993, 1994, 1996"
Burundi,"1993, 1994, 1996"
Kosovo (under UNSC res. 1244),"1998, 1999"
Kuwait,1991
Liberia,"1991, 1992, 1994, 1997"
Rwanda,"1994, 1996"
Somalia,"1991, 1992, 1993"
Syrian Arab Republic,"2012, 2013, 2014"
Timor-Leste,"1999, 2001, 2002"


,Location,Time,NatChangeRT,CNMR,note
10832,Afghanistan,1990,33.41,-38.08,eksen dışı (|net göç| > 20 binde): Savaş dönem...
10842,Afghanistan,2000,38.88,-50.97,eksen dışı (|net göç| > 20 binde): Savaş dönem...
20261,Albania,1995,17.81,-21.42,"eksen dışı (|net göç| > 20 binde): uç değer, n..."
14191,Bahrain,2005,17.10,48.98,eksen dışı (|net göç| > 20 binde): Körfez ülke...
14201,Bahrain,2015,13.28,26.88,eksen dışı (|net göç| > 20 binde): Körfez ülke...
20565,Bosnia and Herzegovina,1995,1.60,34.16,eksen dışı (|net göç| > 20 binde): Bosna Savaşı
65,Burundi,2015,33.57,-21.38,eksen dışı (|net göç| > 20 binde): Burundi siy...
12357,Cambodia,1995,27.96,50.21,"eksen dışı (|net göç| > 20 binde): uç değer, n..."
3541,Chad,1995,33.01,21.47,"eksen dışı (|net göç| > 20 binde): uç değer, n..."
3698,Congo,2000,24.91,25.32,"eksen dışı (|net göç| > 20 binde): uç değer, n..."


**Okuma:** Kriz sıçramalarının çoğu 5 yıllık karelerin arasına düşüyor. Özellikle **Ukrayna 2022** ve Suriye 2012–2014 animasyonda görünmez (kareler 2020, 2025 gibi). Yol haritasındaki "kriz yıllarını açıkla" kuralı yalnızca karelere denk gelen noktalarda uygulanabiliyor. Karar ekibe ait: 5 yıllık düzen korunabilir, ya da 2022 gibi birkaç özel kare eklenebilir (grafik animasyon kare sayısını artırır).

## 5. Çıktılar

In [6]:
cikti = g.rename(columns={"Location": "country", "ISO3_code": "iso3", "Region": "region", "Time": "year",
                          "NatChangeRT": "nat_change_rate", "CNMR": "net_migration_rate"}).copy()
cikti["pop_millions"] = (cikti.TPopulation1July / 1000).round(2)
cikti["nat_change_rate"] = cikti.nat_change_rate.round(2); cikti["net_migration_rate"] = cikti.net_migration_rate.round(2)
cikti["is_projection"] = cikti.year >= 2024
cikti = cikti[["country", "iso3", "region", "year", "nat_change_rate", "net_migration_rate", "pop_millions",
               "pop_change_positive", "quadrant", "is_projection", "note"]].sort_values(["year", "country"]).reset_index(drop=True)

assert len(cikti) == 161 * len(GRID), len(cikti)
assert cikti.drop(columns="note").isna().sum().sum() == 0
assert cikti.groupby("year").country.nunique().eq(161).all()
cikti.to_csv(OUT / "g06_goc_kalkani.csv", index=False)

sayim_out = yillik.reset_index().rename(columns={"Time": "year"})
sayim_out["decline_and_growing_share_pct"] = (sayim_out.n_decline_and_growing / sayim_out.n_natural_decline.replace(0, np.nan) * 100).round(1)
sayim_out.to_csv(OUT / "g06_goc_kalkani_sayim.csv", index=False)
print("Yazıldı:", OUT / "g06_goc_kalkani.csv", cikti.shape); print("Yazıldı:", OUT / "g06_goc_kalkani_sayim.csv", sayim_out.shape)
display(cikti.head(3)); display(cikti.quadrant.value_counts().to_frame("satır"))

Yazıldı: /home/claude/repo/data/out/g06_goc_kalkani.csv (3703, 11)
Yazıldı: /home/claude/repo/data/out/g06_goc_kalkani_sayim.csv (111, 11)


,country,iso3,region,year,nat_change_rate,net_migration_rate,pop_millions,pop_change_positive,quadrant,is_projection,note
0,Afghanistan,AFG,Asia,1990,33.41,-38.08,12.05,False,growth_outflow,False,eksen dışı (|net göç| > 20 binde): Savaş dönem...
1,Albania,ALB,Europe,1990,20.49,-18.36,3.28,True,growth_outflow,False,
2,Algeria,DZA,Africa,1990,24.13,0.08,25.38,True,growth_inflow,False,


,satır
quadrant,
growth_outflow,1572
growth_inflow,859
double_loss,698
shield,574


In [7]:
d = yillik
print(f"BULGU (g06):  2023'te 161 ülkenin {d.loc[2023,'n_natural_decline']}'inde doğal azalış var; yalnızca {d.loc[2023,'n_decline_and_growing']}'i göçle büyümeye devam ediyor.")
print(f"              2050'de {d.loc[2050,'n_natural_decline']} ülkede doğal azalış var, {d.loc[2050,'n_decline_and_growing']}'u göçle büyüyor; 2100'de {d.loc[2100,'n_natural_decline']} ülkede doğal azalış var, yalnızca {d.loc[2100,'n_decline_and_growing']}'ü göçle büyüyor.")
print("NOT:          Doğal azalıştaki ülkeler içinde göçle büyüyenlerin payı %{:.0f} (2023) → %{:.0f} (2050) → %{:.0f} (2100).".format(
      kt.iloc[0]['büyüyen_payı_%'], kt.iloc[1]['büyüyen_payı_%'], kt.iloc[2]['büyüyen_payı_%']))
ok = "Hesaplandı, kontrol bekliyor"
display(pd.DataFrame([(7, "2023'te 35 ülkede doğal azalış, 11'i göçle büyüyor (2050: 61/9, 2100: 101/4)", "Destek", "Lider", ok if kt.tamam.all() else "Fark var")],
                     columns=["#", "sayı", "hesaplayan", "kontrol eden", "durum"]))

BULGU (g06):  2023'te 161 ülkenin 35'inde doğal azalış var; yalnızca 11'i göçle büyümeye devam ediyor.
              2050'de 61 ülkede doğal azalış var, 9'u göçle büyüyor; 2100'de 101 ülkede doğal azalış var, yalnızca 4'ü göçle büyüyor.
NOT:          Doğal azalıştaki ülkeler içinde göçle büyüyenlerin payı %31 (2023) → %15 (2050) → %4 (2100).


,#,sayı,hesaplayan,kontrol eden,durum
0,7,"2023'te 35 ülkede doğal azalış, 11'i göçle büy...",Destek,Lider,"Hesaplandı, kontrol bekliyor"
